# Day 7 — Pattern Mining Engine (Layer 6)

Discover the recurring patterns that separate **funded** from **failed** campaigns, using
association-rule mining (support / confidence / lift) with statistical rigor. Logic lives in
`src/pattern_mining/`; this notebook runs the engine and reads its results.

**Rigor:** every pattern clears support + effect size (lift) + significance (chi-square,
Bonferroni-corrected). **Honesty:** real features only (category, goal, country, timing);
founder/market/revenue/growth are **deferred**. Patterns are historical **associations, not
causes**.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd

# Run the whole engine ONCE (mine-once -> views -> rank -> explain -> cluster -> store).
from src.pattern_mining.pipeline import PatternMiningPipeline
report = PatternMiningPipeline().run()
repo = json.load(open("artifacts/patterns/pattern_repository.json"))
base = report["base_rate"]
print("Engine run. base rate: %.4f | %d campaigns | %d rules mined" %
      (base, report["n_campaigns"], report["n_rules_mined"]))
print("counts:", report["counts"])

Engine run. base rate: 0.4038 | 265171 campaigns | 746 rules mined
counts: {'success': 25, 'failure': 25, 'funding_category_profiles': 15, 'funding_combos': 15, 'temporal_dimensions': 5, 'explanations': 15}


## Phase A — Understanding Pattern Mining

Prediction (Day 4) asks *will this fund?*; retrieval (Day 6) asks *what's similar?*; pattern
mining asks **what repeatedly separates funded from failed?** The measures: **support** (how
common), **confidence** (P(funded | items)), and **lift** = confidence / base rate. Lift is
the key — with a 40% base rate, a 40%-confidence rule is noise. Full notes:
`docs/day7_phaseA_pattern_mining_notes.md`.

## Phase B — Success Pattern Mining

Rules `{items} → funded` with lift > 1, above min support, chi-square significant. Ranked by
lift.

In [2]:
succ = pd.DataFrame(repo["patterns"]["success"])
succ["support_%"] = (succ["support"]*100).round(2)
succ[["label","support_%","confidence","lift","n"]].head(8)

,label,support_%,confidence,lift,n
0,main_category=Comics ∧ goal_band=a very small ...,0.55,0.7748,1.919,1461
1,main_category=Theater ∧ goal_band=a very small...,0.45,0.7631,1.890,1203
2,main_category=Theater ∧ duration_band=<=15d,0.25,0.7290,1.805,668
3,main_category=Theater ∧ goal_band=a small goal,1.45,0.7258,1.797,3836
4,main_category=Dance ∧ goal_band=a small goal,0.54,0.7179,1.778,1425
5,main_category=Theater ∧ country=GB,0.47,0.7060,1.748,1255
6,category=Anthologies,0.22,0.7046,1.745,589
7,main_category=Theater ∧ has_number=has number,0.36,0.7005,1.735,955


## Phase C — Failure Pattern Mining

The mirror: rules over-represented among failures (funding lift < 1). `failure_lift` =
P(failed|items) / P(failed).

In [3]:
fail = pd.DataFrame(repo["patterns"]["failure"])
fail[["label","confidence","lift","failure_lift","n"]].head(8)

,label,confidence,lift,failure_lift,n
0,main_category=Food ∧ goal_band=a very large goal,0.0201,0.050,1.644,997
1,goal_band=a very large goal ∧ duration_band=46...,0.0410,0.101,1.609,2513
2,goal_band=a very large goal ∧ word_count_band=...,0.0538,0.133,1.587,3381
3,category=Apps,0.0694,0.172,1.561,4234
4,goal_band=a very large goal ∧ country=GB,0.0762,0.189,1.550,814
5,goal_band=a very large goal ∧ word_count_band=...,0.0839,0.208,1.537,3220
6,main_category=Film & Video ∧ goal_band=a very ...,0.0872,0.216,1.531,2362
7,goal_band=a large goal ∧ country=IT,0.0875,0.217,1.531,503


## Phase D — Funding Pattern Discovery (goal thresholds)

Per-category **goal thresholds**: funded vs failed median goal. The consistent finding —
winners ask ~half what losers ask, in every category.

In [4]:
prof = pd.DataFrame(repo["patterns"]["funding"]["category_profiles"])
prof[["main_category","success_rate","lift","funded_median_goal_usd","failed_median_goal_usd","best_goal_band"]].head(8)

,main_category,success_rate,lift,funded_median_goal_usd,failed_median_goal_usd,best_goal_band
0,Dance,0.6476,1.604,3000.00,4746.51,a small goal
1,Theater,0.6359,1.575,2627.08,5000.00,a very small goal
2,Comics,0.5867,1.453,2637.27,5000.00,a very small goal
3,Music,0.5276,1.307,3200.00,5000.00,a very small goal
4,Art,0.4517,1.119,2000.00,3852.85,a very small goal
5,Games,0.4395,1.088,5000.00,10000.00,a very small goal
6,Film & Video,0.4175,1.034,5000.00,10000.00,a very small goal
7,Design,0.4156,1.029,7500.00,13000.00,a very small goal


## Phase E — Temporal / Timing Patterns

Honest reframe of "growth" (no time series exists). Funding-rate **trend across launch years**
— the market-saturation story.

In [5]:
yr = pd.DataFrame(repo["patterns"]["temporal"]["year_trend"])
print("Year trend (funding rate over time):")
display(yr[["value","success_rate","lift","n"]])
print("Best weekday:", repo["patterns"]["temporal"]["highlights"].get("best_weekday"),
      "| trend:", repo["patterns"]["temporal"]["highlights"].get("year_trend_direction"))

Year trend (funding rate over time):


,value,success_rate,lift,n
0,2009,0.5026,1.245,951
1,2010,0.4772,1.182,7713
2,2011,0.5065,1.254,19185
3,2012,0.4655,1.153,30841
4,2013,0.4713,1.167,32842
5,2014,0.3551,0.879,47408
6,2015,0.3226,0.799,52240
7,2016,0.3805,0.942,39205
8,2017,0.4244,1.051,34786


Best weekday: Tuesday | trend: declining


## Phase F — Statistical Pattern Ranking (Bonferroni)

Mining many rules manufactures false positives. Bonferroni requires p ≤ α/m. The gap between
"significant at 0.05" and "survives correction" is the honest count of noise.

In [6]:
mt = report["multiple_testing"]
print("rules tested            :", mt["n_rules_tested"])
print("significant at p<=0.05  :", mt["n_raw_significant"])
print("survive Bonferroni      :", mt["n_bonferroni_significant"], "(alpha =", mt["alpha_bonferroni"], ")")
print("REJECTED by correction  :", mt["n_rejected_by_correction"], "  <- would-be false positives")

rules tested            : 746
significant at p<=0.05  : 677
survive Bonferroni      : 610 (alpha = 6.702e-05 )
REJECTED by correction  : 67   <- would-be false positives


## Phase G — Pattern Explanation Generator

Statistical patterns → business-readable sentences, each with the mandatory
association-not-causation caption.

In [7]:
for e in repo["patterns"]["explanations"][:2]:
    print("•", e["explanation"], "\n")

• Historically, campaigns in the Food category and seeking a very large goal were more likely to fail: they funded at only 2% versus a 40% baseline (lift 0.05), across 997 campaigns. This is a historical association in campaign metadata and titles, not a proven cause of funding. 

• Historically, campaigns in the Comics category and seeking a very small goal were more likely to secure funding: they funded at 77% versus a 40% baseline, a lift of 1.92, across 1,461 campaigns. This is a historical association in campaign metadata and titles, not a proven cause of funding. 



## Phase H — Pattern Knowledge Repository

All patterns in one provenance-tagged JSON, with a `query` surface for downstream layers.

In [8]:
from src.pattern_mining.repository import PatternRepository
r = PatternRepository().load()
print("strongest SUCCESS (lift >= 1.7):")
for p in r.query(direction="success", min_lift=1.7, top_k=3): print("  %-46s lift %.2f" % (p["label"][:46], p["lift"]))
print("strongest FAILURE (lift <= 0.15):")
for p in r.query(direction="failure", max_lift=0.15, top_k=3): print("  %-46s lift %.2f" % (p["label"][:46], p["lift"]))
print("deferred capabilities:", len(repo["deferred_capabilities"]))

strongest SUCCESS (lift >= 1.7):
  main_category=Comics ∧ goal_band=a very small  lift 1.92
  main_category=Theater ∧ goal_band=a very small lift 1.89
  main_category=Theater ∧ duration_band=<=15d    lift 1.80
strongest FAILURE (lift <= 0.15):
  main_category=Food ∧ goal_band=a very large go lift 0.05
  goal_band=a very large goal ∧ duration_band=46 lift 0.10
  goal_band=a very large goal ∧ word_count_band= lift 0.13
deferred capabilities: 12


## Phase I — Pipeline + Cluster-based Discovery

The whole engine runs from one call. Clustering (MiniBatchKMeans on the Day-6 embeddings) is
an **unsupervised** view — and it recovers the same success/failure axis the rules found.

In [9]:
cl = pd.DataFrame(report["clusters"])
cl = cl[cl["n"] >= 500]                       # ignore tiny outlier clusters
print("Top success clusters:")
display(cl.head(3)[["cluster","n","success_rate","lift","dominant_main_category","dominant_goal_band"]])
print("Worst (failure) clusters:")
display(cl.tail(2)[["cluster","n","success_rate","lift","dominant_main_category","dominant_goal_band"]])

Top success clusters:


,cluster,n,success_rate,lift,dominant_main_category,dominant_goal_band
0,12,8515,0.6570,1.627,Games,a modest goal
1,19,11303,0.6183,1.531,Theater,a small goal
2,13,19227,0.5969,1.478,Music,a small goal


Worst (failure) clusters:


,cluster,n,success_rate,lift,dominant_main_category,dominant_goal_band
22,17,6890,0.2340,0.579,Film & Video,a modest goal
23,6,5406,0.0975,0.241,Technology,a large goal


## Day 7 complete — Layer 6 ✅

Association-rule mining (support/confidence/lift) → success/failure/funding/temporal patterns
→ Bonferroni-corrected ranking → readable, honesty-captioned explanations → queryable pattern
repository → one-call pipeline with unsupervised cluster corroboration. All real features;
founder/market/revenue/growth deferred. **The story, four ways corroborated: small goals in
creative categories fund; large goals in tech fail.**